In [11]:
import os, random
import pandas as pd

from torch.utils.data import Subset
import numpy as np

from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from PIL import ImageFile, UnidentifiedImageError
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

from torchvision.models import resnet50, ResNet50_Weights

import numpy as np
from sklearn.metrics import f1_score, recall_score, confusion_matrix, classification_report

INPUT_DIR = "/kaggle/input/datasets/shashiprabhapanwar/wikiart"
IMAGES_ROOT = os.path.join(INPUT_DIR, "wikiart_clean", "wikiart")
TRAIN_CSV = os.path.join(INPUT_DIR, "style_train_clean.csv")
VAL_CSV   = os.path.join(INPUT_DIR, "style_val_lite_clean.csv")

In [12]:
def read_wikiart_csv(path):
    df = pd.read_csv(path)

    # headerless got misread as header (common)
    if len(df.columns) == 2:
        c0, c1 = df.columns[0], df.columns[1]
        if ("/" in str(c0)) and str(c1).strip().isdigit():
            df = pd.read_csv(path, header=None, names=["relpath", "label"])
            df["label"] = df["label"].astype(int)
            return df

    # normal header (strip spaces)
    df.columns = [c.strip() for c in df.columns]

    # guess columns
    path_col = df.columns[0]
    label_col = df.columns[1]
    for c in df.columns:
        lc = c.lower()
        if lc in ["pictures", "picture", "path", "file", "filename", "image"]:
            path_col = c
        if lc in ["class", "label", "target"]:
            label_col = c

    df = df[[path_col, label_col]].rename(columns={path_col:"relpath", label_col:"label"})
    df["label"] = df["label"].astype(int)
    return df


def filter_bad_images(df, images_root):
    good_rows = []
    bad = 0

    for relpath, label in zip(df["relpath"].astype(str).values, df["label"].values):
        fullpath = os.path.join(images_root, relpath)
        try:
            with Image.open(fullpath) as im:
                im.verify()  # verifies file integrity without decoding full image
            good_rows.append((relpath, int(label)))
        except (OSError, UnidentifiedImageError):
            bad += 1

    out = pd.DataFrame(good_rows, columns=["relpath", "label"])
    print(f"Filtered bad images: {bad} removed, {len(out)} remaining")
    return out

def evaluate_metrics(model, loader, device, use_amp=True, num_classes=None):
    model.eval()

    all_targets = []
    all_preds = []
    top5_correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                logits = model(x)

            pred = logits.argmax(dim=1)

            # top-1
            total += y.size(0)

            # top-5
            k = min(5, logits.size(1))
            topk = logits.topk(k, dim=1).indices
            top5_correct += (topk == y.unsqueeze(1)).any(dim=1).sum().item()

            all_targets.extend(y.cpu().numpy())
            all_preds.extend(pred.cpu().numpy())

    all_targets = np.array(all_targets)
    all_preds = np.array(all_preds)

    top1 = 100.0 * (all_preds == all_targets).mean()
    top5 = 100.0 * top5_correct / max(1, total)
    macro_f1 = 100.0 * f1_score(all_targets, all_preds, average="macro")
    weighted_f1 = 100.0 * f1_score(all_targets, all_preds, average="weighted")

    if num_classes is None:
        num_classes = int(max(all_targets.max(), all_preds.max())) + 1

    labels = list(range(num_classes))
    per_class_recall = recall_score(
        all_targets,
        all_preds,
        labels=labels,
        average=None,
        zero_division=0
    )

    cm = confusion_matrix(all_targets, all_preds, labels=labels)

    return {
        "top1": top1,
        "top5": top5,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "per_class_recall": per_class_recall,
        "confusion_matrix": cm,
        "targets": all_targets,
        "preds": all_preds,
    }

In [13]:
class WikiArtDataset(Dataset):
    def __init__(self, df, images_root, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.images_root = images_root
        self.transform = transform

        self.df["fullpath"] = self.df["relpath"].astype(str).apply(lambda p: os.path.join(images_root, p))
        # (you already verified 0 missing, so no filtering here)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["fullpath"]
        label = int(row["label"])
    
        img = Image.open(img_path)
        img = img.convert("RGB") 
        if self.transform:
            img = self.transform(img)
        return img, label

class AttnPool1D(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Linear(d, 1)

    def forward(self, x):
        # x: (B, T, D)
        a = self.w(x).squeeze(-1)           # (B, T)
        a = torch.softmax(a, dim=1).unsqueeze(-1)  # (B, T, 1)
        return (x * a).sum(dim=1)           # (B, D)

class ResNet50_FusionCRNN(nn.Module):
    def __init__(self, num_classes, rnn_in=320, rnn_hidden=256, dropout=0.4):
        super().__init__()
        base = resnet50(weights=ResNet50_Weights.DEFAULT)

        self.stem = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4
        )

        self.gap = nn.AdaptiveAvgPool2d(1)

        self.proj = nn.Sequential(
            nn.Conv2d(2048, rnn_in, kernel_size=1, bias=False),
            nn.BatchNorm2d(rnn_in),
            nn.ReLU(inplace=True),
        )

        self.rnn = nn.LSTM(
            input_size=rnn_in,
            hidden_size=rnn_hidden,
            num_layers=2,
            dropout=0.2,
            batch_first=True,
            bidirectional=True,
        )

        self.attn_pool = AttnPool1D(2 * rnn_hidden)
        self.dropout = nn.Dropout(dropout)

        fusion_dim = 2048 + (2 * rnn_hidden) + (2 * rnn_hidden)   # GAP + attn + mean
        self.head = nn.Linear(fusion_dim, num_classes)

    def forward(self, x):
        f = self.stem(x)                      # (B, 2048, H, W)

        gap = self.gap(f).flatten(1)         # (B, 2048)

        z = self.proj(f)                     # (B, rnn_in, H, W)
        seq = z.flatten(2).permute(0, 2, 1) # (B, H*W, rnn_in)

        out, _ = self.rnn(seq)               # (B, T, 2*rnn_hidden)

        attn_vec = self.attn_pool(out)       # (B, 2*rnn_hidden)
        mean_vec = out.mean(dim=1)           # (B, 2*rnn_hidden)

        fused = torch.cat([gap, attn_vec, mean_vec], dim=1)
        fused = self.dropout(fused)

        return self.head(fused)

In [14]:
def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag

def make_optimizer_and_scheduler(model, lr_head=1e-3, lr_backbone=5e-5, wd=1e-4):
    # params for CRNN head (always train these)
    head_params = []
    for name in ["proj", "rnn", "pool", "head", "dropout"]:
        if hasattr(model, name):
            head_params += list(getattr(model, name).parameters())

    # backbone params (stem)
    backbone_params = list(model.stem.parameters())

    # Only include params that are currently trainable
    param_groups = []
    head_params = [p for p in head_params if p.requires_grad]
    backbone_params = [p for p in backbone_params if p.requires_grad]

    if head_params:
        param_groups.append({"params": head_params, "lr": lr_head, "weight_decay": wd})
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": lr_backbone, "weight_decay": wd})

    optimizer = torch.optim.AdamW(param_groups)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.3)
    return optimizer, scheduler

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_df = read_wikiart_csv(TRAIN_CSV)
SUBSET_N = 10000
#train_df = train_df.sample(n=min(SUBSET_N, len(train_df)), random_state=42).reset_index(drop=True)
train_df = filter_bad_images(train_df, IMAGES_ROOT)

val_df = read_wikiart_csv(VAL_CSV)

num_classes = int(max(train_df["label"].max(), val_df["label"].max())) + 1
print("num_classes:", num_classes)
print("train/val sizes:", len(train_df), len(val_df))

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(288, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(288),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_dataset = WikiArtDataset(train_df, IMAGES_ROOT, transform=train_transform)
val_dataset = WikiArtDataset(val_df, IMAGES_ROOT, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False,
                        num_workers=4, pin_memory=True, persistent_workers=True)

# ─── MODEL ───
model = ResNet50_FusionCRNN(num_classes).to(device)

for i in range(4):
    set_requires_grad(model.stem[i], False)
for i in range(4, 8):
    set_requires_grad(model.stem[i], True)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

backbone_params = list(model.stem.parameters())

head_params = (
    list(model.proj.parameters()) +
    list(model.rnn.parameters()) +
    list(model.attn_pool.parameters()) +
    list(model.head.parameters())
)

optimizer = optim.AdamW(
    [
        {"params": head_params, "lr": 8e-4, "weight_decay": 1e-4},
        {"params": backbone_params, "lr": 2e-4, "weight_decay": 1e-4},
    ]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=15,
    eta_min=1e-6
)

def topk_acc(logits, y, k=5):
    k = min(k, logits.size(1))
    _, topk = logits.topk(k, dim=1)
    return (topk == y.unsqueeze(1)).any(dim=1).float().mean().item()

# Mixup, used for mixing up two pic with different classes like x% class A and y% class B to train model on soft probablity 
def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

epochs = 15
best_top1 = -1.0

for epoch in range(1, epochs + 1):
    model.train()
    loss_sum = 0.0
    steps = 0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item()
        steps += 1

    scheduler.step()

    metrics = evaluate_metrics(
        model=model,
        loader=val_loader,
        device=device,
        use_amp=use_amp,
        num_classes=num_classes
    )

    print(
        f"Epoch {epoch:02d}/{epochs} | "
        f"train_loss {loss_sum/max(1,steps):.4f} | "
        f"val_top1 {metrics['top1']:.2f}% | "
        f"val_top5 {metrics['top5']:.2f}% | "
        f"macro_f1 {metrics['macro_f1']:.2f}% | "
        f"weighted_f1 {metrics['weighted_f1']:.2f}%"
    )

    if metrics["top1"] > best_top1:
        best_top1 = metrics["top1"]
        torch.save(model.state_dict(), "best_style_crnn.pt")
        print(f"Saved best checkpoint with top1={best_top1:.2f}%")

final_metrics = evaluate_metrics(
    model=model,
    loader=val_loader,
    device=device,
    use_amp=use_amp,
    num_classes=num_classes
)

print("\n================ FINAL VALIDATION METRICS ================\n")

print(f"Top-1 Accuracy    : {final_metrics['top1']:.2f}%")
print(f"Top-5 Accuracy    : {final_metrics['top5']:.2f}%")
print(f"Macro F1 Score    : {final_metrics['macro_f1']:.2f}%")
print(f"Weighted F1 Score : {final_metrics['weighted_f1']:.2f}%")

print("\nPer-Class Recall:\n")

for i, r in enumerate(final_metrics["per_class_recall"]):
    print(f"Class {i:02d} : {100*r:.2f}%")

from sklearn.metrics import classification_report

print("\nClassification Report:\n")

print(
    classification_report(
        final_metrics["targets"],
        final_metrics["preds"],
        digits=3
    )
)

import seaborn as sns
import matplotlib.pyplot as plt

cm = final_metrics["confusion_matrix"]

plt.figure(figsize=(10,8))
sns.heatmap(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

Device: cuda
Filtered bad images: 0 removed, 57025 remaining
num_classes: 27
train/val sizes: 57025 10178
Epoch 01/15 | train_loss 1.6856 | val_top1 56.25% | val_top5 91.40% | macro_f1 53.41% | weighted_f1 56.56%
Saved best checkpoint with top1=56.25%
Epoch 02/15 | train_loss 1.3145 | val_top1 58.80% | val_top5 92.12% | macro_f1 57.35% | weighted_f1 59.16%
Saved best checkpoint with top1=58.80%
Epoch 03/15 | train_loss 1.1356 | val_top1 61.25% | val_top5 93.96% | macro_f1 61.43% | weighted_f1 62.05%
Saved best checkpoint with top1=61.25%
Epoch 04/15 | train_loss 0.9963 | val_top1 62.35% | val_top5 94.07% | macro_f1 61.63% | weighted_f1 62.50%
Saved best checkpoint with top1=62.35%
Epoch 05/15 | train_loss 0.8642 | val_top1 64.54% | val_top5 94.56% | macro_f1 64.09% | weighted_f1 64.82%
Saved best checkpoint with top1=64.54%
Epoch 06/15 | train_loss 0.7633 | val_top1 64.32% | val_top5 94.18% | macro_f1 63.86% | weighted_f1 64.54%
Epoch 07/15 | train_loss 0.6798 | val_top1 64.41% | val_t

# 